# Projeto: Chatbot Guia Virtual de Penedo (RAG) 🏰

Este notebook implementa um **Chatbot com RAG** capaz de responder perguntas sobre a cidade histórica de Penedo-AL.
O sistema utiliza um documento PDF como base de conhecimento, garantindo que as respostas sejam fundamentadas em fatos históricos e turísticos reais, evitando alucinações da IA.

## 📚 Fonte de Dados (Créditos)
O documento base utilizado neste projeto (`penedo.pdf`) é derivado do artigo científico:
> **Título:** As riquezas de Penedo - Alagoas/Brasil  
> **Autor:** Thiago Pereira Dantas  
> **Publicação:** Revista Científica Multidisciplinar Núcleo do Conhecimento. Ano 06, Ed. 03, Vol. 08, pp. 165-190. Março de 2021.  
> **Link:** [Acessar Artigo Original](https://www.nucleodoconhecimento.com.br/historia/riquezas-de-penedo)

## 🛠️ Tecnologias Utilizadas
- **LangChain:** Orquestração do fluxo de RAG.
- **ChromaDB:** Banco de dados vetorial para busca semântica.
- **OpenAI (via OpenRouter):** LLM (GPT-4o-mini) e Embeddings.
- **Gradio:** Interface gráfica.

In [ ]:
# ==========================================
# ETAPA 1: Configuração do Ambiente
# ==========================================

# Instalação das dependências
print("⏳ Instalando bibliotecas...")
!pip install -q langchain langchain-community langchain-openai openai chromadb pypdf tiktoken gradio python-dotenv \
"requests==2.32.4" "opentelemetry-api==1.37.0" "opentelemetry-sdk==1.37.0" \
"opentelemetry-exporter-otlp-proto-common==1.37.0" "opentelemetry-proto==1.37.0"

import os
from getpass import getpass

print("\n✅ Instalação concluída!")
# Configuração da chave API da OpenRouter
if "OPENROUTER_API_KEY" not in os.environ:
    api_key = getpass("🔑 Digite a OPENROUTER_API_KEY aqui e pressione Enter: ")
    os.environ["OPENROUTER_API_KEY"] = api_key
    print("✅ Chave de API configurada.")
else:
    print("✅ Chave de API já configurada no ambiente.")

In [ ]:
# ==============================================
# ETAPA 2: Validação da Base de Conhecimento
# ==============================================
import os

pdf_filename = "penedo.pdf"

# Verifica se o arquivo PDF existe no diretório atual
if not os.path.exists(pdf_filename):
    raise FileNotFoundError(f"❌ ERRO: O arquivo '{pdf_filename}' não foi encontrado no diretório. Por favor, adicione-o antes de continuar.")
else:
    print(f"✅ Arquivo '{pdf_filename}' carregado com sucesso!")

In [ ]:
# ==========================================
# ETAPA 3: Construção do Motor de RAG
# ==========================================
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

def setup_rag_pipeline(pdf_path):
    print(f"⚙️ Iniciando processamento de IA para: {pdf_path}")

    # 1. Carregamento -> Lê o arquivo PDF e extrai o texto bruto.
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    print(f"   📄 PDF lido: {len(docs)} páginas encontradas.")

    # 2. Fragmentação -> Divide o texto em pedaços menores para caber no contexto da IA e facilitar a busca.
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = text_splitter.split_documents(docs)
    print(f"   ✂️ Texto fragmentado em {len(chunks)} blocos de informação.")

    # 3. Vetorização (Embeddings) -> Converte os textos em números (vetores) para busca semântica.
    base_url = "https://openrouter.ai/api/v1"
    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small",
        api_key=os.environ["OPENROUTER_API_KEY"],
        base_url=base_url
    )

    # 4. Banco Vetorial -> Armazena os vetores em memória usando ChromaDB.
    vectorstore = Chroma.from_documents(
        documents = chunks,
        embedding = embeddings
    )
    print("   🗄️ Banco de dados vetorial construído na memória.")

    # 5. Modelo de Linguagem
    llm = ChatOpenAI(
        model = "openai/gpt-4o-mini",
        api_key = os.environ["OPENROUTER_API_KEY"],
        base_url = base_url,
        temperature = 0.7
    )

    # 6. Define a persona e regras estritas para usar apenas o contexto.
    prompt_template = """
    Você é o "Guia Penedo", um assistente turístico virtual especializado na cidade histórica de Penedo, Alagoas.

    SUAS INSTRUÇÕES:
    1. Use APENAS o contexto fornecido abaixo para responder.
    2. Seja educado, acolhedor e convide o turista a conhecer a cidade.
    3. Se a informação não estiver no contexto, responda: "Desculpe, não tenho essa informação nos meus registros históricos de Penedo no momento."

    CONTEXTO:
    {context}

    PERGUNTA DO USUÁRIO: {question}

    RESPOSTA:
    """
    PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

    # 7. Cadeia de Recuperação (Chain)
    qa_chain = RetrievalQA.from_chain_type(
        llm = llm,
        chain_type = "stuff",
        retriever = vectorstore.as_retriever(),
        chain_type_kwargs = {"prompt": PROMPT}
    )

    return qa_chain

# Inicializa a IA
qa_pipeline = setup_rag_pipeline(pdf_filename)
print("\n🤖 MOTOR DE IA PRONTO PARA USO!")

In [ ]:
# ================================================
# ETAPA 4 (Opção A): Interface Gráfica com Gradio
# ================================================
import gradio as gr

def responder_turista(message, history):
    if not message:
        return ""

    # Chama o pipeline criado anteriormente
    resultado = qa_pipeline.invoke({"query": message})
    return resultado['result']

# Configuração da Interface
interface = gr.ChatInterface(
    fn=responder_turista,
    title = "🏰 Guia Virtual de Penedo - AL",
    description = "Pergunte sobre a história, igrejas, cultura e turismo de Penedo. Eu respondo com base em documentos oficiais!",
    theme = "soft",
    examples = ["Quem fundou Penedo?", "Quais as igrejas mais famosas?", "Onde fica o Teatro Sete de Setembro?", "Fale sobre o Rio São Francisco"]
)

print("🚀 Iniciando interface gráfica...")
interface.launch(share=True, debug=True)

In [ ]:
# ==========================================
# ETAPA 4 (Opção B): Interface via Terminal
# ==========================================

print("==================================================")
print("🤖 CHATBOT INICIADO (Modo Terminal)")
print("Digite 'sair' para encerrar.")
print("==================================================\n")

while True:
    user_input = input("VOCÊ: ")

    if user_input.lower() in ['sair', 'exit', 'fim']:
        print("👋 Até logo!")
        break

    if not user_input.strip():
        continue

    print("⏳ Pesquisando...", end="\r")
    response = qa_pipeline.invoke({"query": user_input})

    print(f"GUIA PENEDO: {response['result']}")
    print("-" * 50)